라이브러리 불러오기

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

Google 계정 인증

In [2]:
from google.colab import auth

auth.authenticate_user()
print("Google 계정 인증 완료")

Google 계정 인증 완료


BigQuery 데이터 불러오기

In [3]:
from google.cloud import bigquery

PROJECT_ID = "landing-page-ab-test"

client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT *
FROM `landing-page-ab-test.landing_page_ab_test.ab_test_clean`
"""

df = client.query(query).to_dataframe()

print(df.shape)
df.head()

(294478, 16)


,user_id,timestamp,event_date,event_month,group,landing_page,converted,is_converter,age,age_group,gender,location,session_duration,pages_visited,device_type,purchase_amount
0,U1446,2024-10-30 10:57:38.967303+00:00,2024-10-30,2024-10-01,control,old_page,0,Non-converter,18,10대,Female,UK,6.13,4,Desktop,0.0
1,U4608,2025-03-31 02:58:40.656342+00:00,2025-03-31,2025-03-01,control,old_page,0,Non-converter,18,10대,Male,US,5.34,4,Desktop,0.0
2,U5348,2024-08-19 05:15:39.204705+00:00,2024-08-19,2024-08-01,control,old_page,0,Non-converter,18,10대,Female,Pakistan,4.49,4,Mobile,0.0
3,U6944,2024-01-11 03:45:14.800634+00:00,2024-01-11,2024-01-01,control,old_page,0,Non-converter,18,10대,Male,Australia,6.66,3,Desktop,0.0
4,U9439,2024-01-16 18:10:28.832645+00:00,2024-01-16,2024-01-01,control,old_page,0,Non-converter,18,10대,Male,US,5.34,8,Desktop,0.0


기본 구조 확인

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   user_id           294478 non-null  object             
 1   timestamp         294478 non-null  datetime64[us, UTC]
 2   event_date        294478 non-null  dbdate             
 3   event_month       294478 non-null  dbdate             
 4   group             294478 non-null  object             
 5   landing_page      294478 non-null  object             
 6   converted         294478 non-null  Int64              
 7   is_converter      294478 non-null  object             
 8   age               294478 non-null  Int64              
 9   age_group         294478 non-null  object             
 10  gender            294478 non-null  object             
 11  location          294478 non-null  object             
 12  session_duration  294478 non-null  float64  

In [5]:
df[[
    "group",
    "landing_page",
    "converted",
    "age_group",
    "gender",
    "location",
    "device_type"
]].describe(include="all")

,group,landing_page,converted,age_group,gender,location,device_type
count,294478,294478,294478.0,294478,294478,294478,294478
unique,2,2,<NA>,6,3,7,3
top,treatment,new_page,<NA>,30대,Male,US,Desktop
freq,147552,147552,<NA>,108055,144708,88342,176692
mean,NaN,NaN,0.149172,NaN,NaN,NaN,NaN
std,NaN,NaN,0.356259,NaN,NaN,NaN,NaN
min,NaN,NaN,0.0,NaN,NaN,NaN,NaN
25%,NaN,NaN,0.0,NaN,NaN,NaN,NaN
50%,NaN,NaN,0.0,NaN,NaN,NaN,NaN
75%,NaN,NaN,0.0,NaN,NaN,NaN,NaN


그룹별 전환 수와 표본 수 확인

In [6]:
summary = (
    df.groupby("group")
      .agg(
          users=("user_id", "nunique"),
          conversions=("converted", "sum")
      )
)

summary["conversion_rate"] = (
    summary["conversions"] / summary["users"]
)

summary

,users,conversions,conversion_rate
group,,,
control,146926,17444,0.118726
treatment,147552,26484,0.179489


Two-Proportion Z-test

In [7]:
control_success = summary.loc["control", "conversions"]
treatment_success = summary.loc["treatment", "conversions"]

control_n = summary.loc["control", "users"]
treatment_n = summary.loc["treatment", "users"]

count = np.array([
    treatment_success,
    control_success
])

nobs = np.array([
    treatment_n,
    control_n
])

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative="two-sided"
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.10f}")

Z-statistic: 46.2773
P-value: 0.0000000000


전환율 차이와 95% 신뢰구간

In [8]:
control_rate = control_success / control_n
treatment_rate = treatment_success / treatment_n

absolute_lift = treatment_rate - control_rate
relative_lift = absolute_lift / control_rate

standard_error = np.sqrt(
    treatment_rate * (1 - treatment_rate) / treatment_n
    + control_rate * (1 - control_rate) / control_n
)

ci_lower = absolute_lift - 1.96 * standard_error
ci_upper = absolute_lift + 1.96 * standard_error

print(f"Control 전환율: {control_rate:.4%}")
print(f"Treatment 전환율: {treatment_rate:.4%}")
print(f"절대 전환율 차이: {absolute_lift:.4%}")
print(f"상대 개선율: {relative_lift:.2%}")
print(
    f"95% 신뢰구간: "
    f"[{ci_lower:.4%}, {ci_upper:.4%}]"
)

Control 전환율: 11.8726%
Treatment 전환율: 17.9489%
절대 전환율 차이: 6.0763%
상대 개선율: 51.18%
95% 신뢰구간: [5.8200%, 6.3326%]


효과 크기와 NNT

In [9]:
cohen_h = (
    2 * np.arcsin(np.sqrt(treatment_rate))
    - 2 * np.arcsin(np.sqrt(control_rate))
)

nnt = 1 / absolute_lift

print(f"Cohen's h: {cohen_h:.4f}")
print(f"추가 전환 1건당 필요 사용자 수: {nnt:.2f}명")

Cohen's h: 0.1714
추가 전환 1건당 필요 사용자 수: 16.46명


세션 지속시간, 방문 페이지 수, 구매자 구매금액 검정

In [10]:
# 그룹별 분석 데이터 준비
control = df[df["group"] == "control"]
treatment = df[df["group"] == "treatment"]

# 구매금액은 실제 구매자만 비교
control_buyers = control[control["converted"] == 1]
treatment_buyers = treatment[treatment["converted"] == 1]


def compare_groups(control_values, treatment_values, metric_name):
    """Welch t-test와 Mann–Whitney U test를 함께 수행합니다."""

    control_values = control_values.astype(float).dropna()
    treatment_values = treatment_values.astype(float).dropna()

    # 평균 차이 검정
    t_stat, t_pvalue = stats.ttest_ind(
        treatment_values,
        control_values,
        equal_var=False
    )

    # 분포 차이 검정
    u_stat, u_pvalue = stats.mannwhitneyu(
        treatment_values,
        control_values,
        alternative="two-sided"
    )

    # Cohen's d
    pooled_sd = np.sqrt(
        (
            control_values.var(ddof=1)
            + treatment_values.var(ddof=1)
        ) / 2
    )

    cohens_d = (
        treatment_values.mean() - control_values.mean()
    ) / pooled_sd

    return {
        "지표": metric_name,
        "Control 평균": control_values.mean(),
        "Treatment 평균": treatment_values.mean(),
        "평균 차이": treatment_values.mean() - control_values.mean(),
        "Welch t-test p-value": t_pvalue,
        "Mann-Whitney p-value": u_pvalue,
        "Cohen's d": cohens_d
    }


test_results = pd.DataFrame([
    compare_groups(
        control["session_duration"],
        treatment["session_duration"],
        "세션 지속시간"
    ),
    compare_groups(
        control["pages_visited"],
        treatment["pages_visited"],
        "방문 페이지 수"
    ),
    compare_groups(
        control_buyers["purchase_amount"],
        treatment_buyers["purchase_amount"],
        "구매자 구매금액"
    )
])

test_results

,지표,Control 평균,Treatment 평균,평균 차이,Welch t-test p-value,Mann-Whitney p-value,Cohen's d
0,세션 지속시간,5.003684,5.000594,-0.003090,0.672041,0.469557,-0.001560
1,방문 페이지 수,4.015327,4.023754,0.008427,0.244635,0.159050,0.004288
2,구매자 구매금액,37.504446,37.680802,0.176356,0.362207,0.243005,0.008884


예상 추가 전환 수와 예상 추가 매출 계산

In [11]:
# 기본 값
control_rate = summary.loc["control", "conversion_rate"]
treatment_rate = summary.loc["treatment", "conversion_rate"]
treatment_users = summary.loc["treatment", "users"]
treatment_conversions = summary.loc["treatment", "conversions"]

avg_purchase_amount_treatment = (
    treatment_buyers["purchase_amount"].mean()
)

# Control 전환율이 유지됐을 경우 예상 전환 수
expected_conversions_at_control_rate = (
    treatment_users * control_rate
)

# 신규 페이지로 인해 발생한 것으로 추정되는 추가 전환 수
incremental_conversions = (
    treatment_conversions
    - expected_conversions_at_control_rate
)

# 추가 매출 추정
incremental_revenue = (
    incremental_conversions
    * avg_purchase_amount_treatment
)

print(f"Control 전환율 기준 예상 전환 수: {expected_conversions_at_control_rate:,.0f}건")
print(f"실제 Treatment 전환 수: {treatment_conversions:,.0f}건")
print(f"예상 추가 전환 수: {incremental_conversions:,.0f}건")
print(f"Treatment 구매자당 평균 구매금액: {avg_purchase_amount_treatment:,.2f}")
print(f"예상 추가 매출: {incremental_revenue:,.2f}")

Control 전환율 기준 예상 전환 수: 17,518건
실제 Treatment 전환 수: 26,484건
예상 추가 전환 수: 8,966건
Treatment 구매자당 평균 구매금액: 37.68
예상 추가 매출: 337,833.91
